# Ecuaciones diferenciales ordinarias I
## Método de Euler, Euler mejorado y análisis del error

**Computación Avanzada y Analítica**

Este notebook acompaña la clase. Contiene las implementaciones iniciales y los ejemplos resueltos. La idea es mantener una correspondencia directa entre las fórmulas matemáticas y el código.

## 1. Problema de valor inicial

Trabajaremos con problemas de la forma

$$y'=f(t,y),\qquad y(t_0)=y_0.$$

Una aproximación numérica construye valores $y_n\approx y(t_n)$ sobre una malla $t_n=t_0+nh$.

## 2. Método de Euler

La fórmula es

$$y_{n+1}=y_n+h f(t_n,y_n).$$

En cada iteración usamos la pendiente en el punto actual y avanzamos un paso $h$.

In [1]:
def euler(f, t0, y0, h, n):
    """Método de Euler para un PVI y' = f(t,y)."""
    t = t0
    y = y0
    valores = [(t, y)]

    for i in range(n):
        y = y + h * f(t, y)
        t = t + h
        valores.append((t, y))

    return valores

In [13]:
help(euler)

Help on function euler in module __main__:

euler(f, t0, y0, h, n)
    Método de Euler para un PVI y' = f(t,y).



## 3. Primer ejemplo: $y'=y$, $y(0)=1$

La solución exacta es $y(t)=e^t$. Usaremos $h=0.1$ hasta $t=0.5$.

In [2]:
import math

def f(t, y):
    return y

sol_euler = euler(f, 0, 1, 0.1, 5)

for t, y in sol_euler:
    print(f"t = {t:.1f}, y = {y:.6f}")

t = 0.0, y = 1.000000
t = 0.1, y = 1.100000
t = 0.2, y = 1.210000
t = 0.3, y = 1.331000
t = 0.4, y = 1.464100
t = 0.5, y = 1.610510


In [3]:
y_exacta = math.exp(0.5)
y_euler = sol_euler[-1][1]
error_abs = abs(y_exacta - y_euler)
error_rel = error_abs / abs(y_exacta)

print(f"Exacta: {y_exacta:.6f}")
print(f"Euler:  {y_euler:.6f}")
print(f"Error absoluto: {error_abs:.6f}")
print(f"Error relativo: {error_rel:.6%}")

Exacta: 1.648721
Euler:  1.610510
Error absoluto: 0.038211
Error relativo: 2.317631%


## 4. Método de Euler mejorado

Primero hacemos una predicción de Euler:

$$y_{n+1}^{*}=y_n+h f(t_n,y_n).$$

Luego promediamos la pendiente inicial y la pendiente en el punto predicho:

$$y_{n+1}=y_n+\frac{h}{2}\left[f(t_n,y_n)+f(t_{n+1},y_{n+1}^{*})\right].$$

In [4]:
def euler_mejorado(f, t0, y0, h, n):
    """Método de Euler mejorado (predictor-corrector)."""
    t = t0
    y = y0
    valores = [(t, y)]

    for i in range(n):
        # Predicción de Euler
        y_pred = y + h * f(t, y)

        # Corrección usando el promedio de pendientes
        y = y + (h / 2) * (
            f(t, y) + f(t + h, y_pred)
        )

        t = t + h
        valores.append((t, y))

    return valores

In [14]:
sol_mejorado = euler_mejorado(f, 0, 1, 0.1, 5)

for (t1, y1), (_, y2) in zip(sol_euler, sol_mejorado):
    exacta = math.exp(t1)
    print(f"t={t1:.1f} | Euler={y1:.6f} | Euler mejorado={y2:.6f} | Exacta={exacta:.6f}")

t=0.0 | Euler=1.000000 | Euler mejorado=1.000000 | Exacta=1.000000
t=0.1 | Euler=1.100000 | Euler mejorado=1.105000 | Exacta=1.105171
t=0.2 | Euler=1.210000 | Euler mejorado=1.221025 | Exacta=1.221403
t=0.3 | Euler=1.331000 | Euler mejorado=1.349233 | Exacta=1.349859
t=0.4 | Euler=1.464100 | Euler mejorado=1.490902 | Exacta=1.491825
t=0.5 | Euler=1.610510 | Euler mejorado=1.647447 | Exacta=1.648721


In [6]:
y_mejorado = sol_mejorado[-1][1]
error_mejorado = abs(y_exacta - y_mejorado)

print(f"Error Euler:           {error_abs:.6f}")
print(f"Error Euler mejorado:  {error_mejorado:.6f}")

Error Euler:           0.038211
Error Euler mejorado:  0.001275


## 5. Función para calcular errores

El error absoluto es $|y_{exacta}-y_{aprox}|$ y el relativo es $|y_{exacta}-y_{aprox}|/|y_{exacta}|$.

In [7]:
def errores(y_exacta, y_aprox):
    error_abs = abs(y_exacta - y_aprox)
    error_rel = error_abs / abs(y_exacta)
    return error_abs, error_rel

print(errores(y_exacta, y_euler))

(0.038211270700128086, 0.023176307226206706)


## 6. Segundo ejemplo de clase: $y'=t+y$, $y(0)=1$

Para la práctica podemos calcular la solución exacta con SymPy y luego comparar ambos métodos.

In [8]:
import sympy as sp

t = sp.symbols('t')
y = sp.Function('y')

solucion = sp.dsolve(
    sp.Eq(sp.diff(y(t), t), t + y(t)),
    ics={y(0): 1}
)

solucion

Eq(y(t), -t + 2*exp(t) - 1)

In [9]:
f2 = lambda t, y: t + y

sol2_euler = euler(f2, 0, 1, 0.1, 5)
sol2_mejorado = euler_mejorado(f2, 0, 1, 0.1, 5)

for (t1, ye), (_, ym) in zip(sol2_euler, sol2_mejorado):
    exacta = float(solucion.rhs.subs(t, t1))
    print(f"t={t1:.1f} | Euler={ye:.6f} | Mejorado={ym:.6f} | Exacta={exacta:.6f}")

t=0.0 | Euler=1.000000 | Mejorado=1.000000 | Exacta=1.000000
t=0.1 | Euler=1.100000 | Mejorado=1.110000 | Exacta=1.110342
t=0.2 | Euler=1.220000 | Mejorado=1.242050 | Exacta=1.242806
t=0.3 | Euler=1.362000 | Mejorado=1.398465 | Exacta=1.399718
t=0.4 | Euler=1.528200 | Mejorado=1.581804 | Exacta=1.583649
t=0.5 | Euler=1.721020 | Mejorado=1.794894 | Exacta=1.797443


## 7. Segundo ejercicio: análisis del tamaño de paso

Consideramos

$$y'=-2y+t,\qquad y(0)=1.$$

La idea es probar varios tamaños de paso y observar cómo cambia el error en $t=2$.

In [10]:
f3 = lambda t, y: -2*y + t

# Solución exacta obtenida previamente de forma simbólica
solucion3 = sp.dsolve(
    sp.Eq(sp.diff(y(t), t), -2*y(t) + t),
    ics={y(0): 1}
)

solucion3

Eq(y(t), t/2 - 1/4 + 5*exp(-2*t)/4)

In [11]:
hs = [0.2, 0.1, 0.05, 0.01]
y_exacta_2 = float(solucion3.rhs.subs(t, 2))

print(f"Valor exacto en t=2: {y_exacta_2:.10f}\n")

for h in hs:
    n = round(2 / h)
    e = euler(f3, 0, 1, h, n)[-1][1]
    em = euler_mejorado(f3, 0, 1, h, n)[-1][1]
    ee = abs(y_exacta_2 - e)
    eem = abs(y_exacta_2 - em)
    print(f"h={h:<5} | Euler error={ee:.8e} | Mejorado error={eem:.8e}")

Valor exacto en t=2: 0.7728945486

h=0.2   | Euler error=1.53362766e-02 | Mejorado error=3.52948664e-03
h=0.1   | Euler error=8.48302980e-03 | Mejorado error=7.20403037e-04
h=0.05  | Euler error=4.41844493e-03 | Mejorado error=1.65126341e-04
h=0.01  | Euler error=9.09615354e-04 | Mejorado error=6.19836258e-06


## 8. Comprobación experimental del orden

Si $E(h)\approx Ch^p$, entonces

$$p\approx\frac{\log(E(h)/E(h/2))}{\log 2}.$$

Para Euler esperamos $p\approx1$ y para Euler mejorado $p\approx2$.

In [12]:
def estimar_orden(e_h, e_h2):
    return math.log(e_h / e_h2, 2)

# Errores para h=0.2 y h=0.1
e_02 = abs(y_exacta_2 - euler(f3, 0, 1, 0.2, 10)[-1][1])
e_01 = abs(y_exacta_2 - euler(f3, 0, 1, 0.1, 20)[-1][1])
em_02 = abs(y_exacta_2 - euler_mejorado(f3, 0, 1, 0.2, 10)[-1][1])
em_01 = abs(y_exacta_2 - euler_mejorado(f3, 0, 1, 0.1, 20)[-1][1])

print("Orden estimado Euler:", estimar_orden(e_02, e_01))
print("Orden estimado Euler mejorado:", estimar_orden(em_02, em_01))

Orden estimado Euler: 0.8542967276986998
Orden estimado Euler mejorado: 2.2925821929310177


## 9. Ideas para continuar

- Visualizar las soluciones exacta y numéricas.
- Automatizar tablas de resultados.
- Comparar distintos tamaños de paso.
- Estudiar estabilidad numérica.
- Extender el procedimiento a sistemas de EDO.

### Ejercicios para casa
1. Resolver $y'=t+y$, $y(0)=1$ con ambos métodos y $h=0.1$.
2. Para $y'=-2y+t$, $y(0)=1$, comparar $h=0.2,0.1,0.05,0.01$ y verificar los órdenes experimentales.